# 📊 Customer Churn Prediction
**Author:** Miral Butt | Data Analyst | MPhil Statistics

---

## 🎯 Project Overview
Customer churn means when customers stop using a company's service. This project analyzes customer data to:
- Understand what factors lead to churn
- Build a Machine Learning model to predict churn
- Provide actionable business recommendations

**Tools Used:** Python, Pandas, Matplotlib, Seaborn, Scikit-learn

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('✅ Libraries loaded successfully!')

## 2. Load & Explore Dataset
We use a telecom customer dataset with 1,000 customers and features like contract type, monthly charges, tenure, and support calls.

In [ ]:
np.random.seed(42)
n = 1000

tenure         = np.random.randint(1, 72, n)
monthly_charges= np.round(np.random.uniform(20, 120, n), 2)
contract       = np.random.choice(['Month-to-Month', 'One Year', 'Two Year'],
                                   n, p=[0.55, 0.25, 0.20])
support_calls  = np.random.poisson(2, n)
internet       = np.random.choice(['DSL', 'Fiber Optic', 'No'], n, p=[0.35, 0.45, 0.20])
senior         = np.random.choice([0, 1], n, p=[0.84, 0.16])

# Churn logic: short tenure + high charges + month-to-month => more likely to churn
churn_prob = (
    0.3
    + (tenure < 12) * 0.25
    + (monthly_charges > 80) * 0.20
    + (contract == 'Month-to-Month') * 0.20
    + (support_calls > 3) * 0.10
    - (contract == 'Two Year') * 0.25
)
churn_prob = np.clip(churn_prob, 0, 1)
churn      = (np.random.rand(n) < churn_prob).astype(int)

df = pd.DataFrame({
    'CustomerID'      : [f'CUST-{i:04d}' for i in range(1, n+1)],
    'Tenure_Months'   : tenure,
    'Monthly_Charges' : monthly_charges,
    'Contract_Type'   : contract,
    'Internet_Service': internet,
    'Support_Calls'   : support_calls,
    'Senior_Citizen'  : senior,
    'Churn'           : churn
})

print(f'Dataset Shape: {df.shape}')
print(f'\nChurn Rate: {df["Churn"].mean()*100:.1f}%')
df.head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Customer Churn - Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)

# 1. Churn distribution
churn_counts = df['Churn'].value_counts()
axes[0,0].pie(churn_counts, labels=['No Churn', 'Churned'],
              autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'], startangle=90)
axes[0,0].set_title('Overall Churn Distribution', fontweight='bold')

# 2. Churn by Contract Type
contract_churn = df.groupby('Contract_Type')['Churn'].mean() * 100
contract_churn.plot(kind='bar', ax=axes[0,1], color=['#3498db','#e67e22','#9b59b6'], edgecolor='white')
axes[0,1].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0,1].set_ylabel('Churn Rate (%)')
axes[0,1].tick_params(axis='x', rotation=15)
for bar in axes[0,1].patches:
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                   f'{bar.get_height():.1f}%', ha='center', fontsize=9)

# 3. Tenure vs Churn
df[df['Churn']==0]['Tenure_Months'].hist(ax=axes[1,0], alpha=0.6, label='No Churn', color='#2ecc71', bins=20)
df[df['Churn']==1]['Tenure_Months'].hist(ax=axes[1,0], alpha=0.6, label='Churned', color='#e74c3c', bins=20)
axes[1,0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[1,0].set_xlabel('Tenure (Months)')
axes[1,0].legend()

# 4. Monthly Charges vs Churn
df.boxplot(column='Monthly_Charges', by='Churn', ax=axes[1,1],
           patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.6))
axes[1,1].set_title('Monthly Charges by Churn', fontweight='bold')
axes[1,1].set_xlabel('Churn (0=No, 1=Yes)')
axes[1,1].set_ylabel('Monthly Charges ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved!')

## 4. Key Insights from EDA

| Finding | Insight |
|---|---|
| **Contract Type** | Month-to-Month customers churn the most |
| **Tenure** | New customers (< 12 months) are at highest risk |
| **Monthly Charges** | Higher charges correlate with more churn |
| **Support Calls** | More calls = more frustration = more churn |

## 5. Data Preprocessing

In [ ]:
df_model = df.drop('CustomerID', axis=1)

# Encode categorical columns
le = LabelEncoder()
df_model['Contract_Type']    = le.fit_transform(df_model['Contract_Type'])
df_model['Internet_Service'] = le.fit_transform(df_model['Internet_Service'])

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print('✅ Preprocessing complete!')

## 6. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'pred': y_pred, 'accuracy': acc}
    print(f'\n=== {name} ===')
    print(f'Accuracy: {acc*100:.2f}%')
    print(classification_report(y_test, y_pred, target_names=['No Churn','Churned']))

In [ ]:
# Confusion Matrix & Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Results', fontsize=15, fontweight='bold')

# Confusion matrix - Random Forest
cm = confusion_matrix(y_test, results['Random Forest']['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn','Churned'], yticklabels=['No Churn','Churned'])
axes[0].set_title('Random Forest - Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature importance
rf = results['Random Forest']['model']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Feature Importance (Random Forest)', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model results saved!')

## 7. Conclusions & Business Recommendations

### 📈 Model Performance
- **Random Forest** outperforms Logistic Regression
- Model successfully identifies high-risk customers before they churn

### 💡 Business Recommendations

| Priority | Action |
|---|---|
| 🔴 High | Offer discounts to Month-to-Month customers to switch to annual plans |
| 🔴 High | Create special retention program for customers in first 12 months |
| 🟡 Medium | Proactively reach customers with 3+ support calls |
| 🟡 Medium | Review pricing strategy for high monthly charge segments |
| 🟢 Low | Use ML model to score all customers monthly and flag at-risk ones |

### 🔮 Next Steps
- Deploy model as an API for real-time churn scoring
- A/B test retention offers on predicted churn segments
- Collect more features (e.g., satisfaction scores, usage patterns)